In [10]:
import numpy as np
import pandas as pd
import pickle, time

In [11]:
PATH = "data/train_triplets.txt"

In [12]:
user_map, song_map = {}, {}

In [13]:
def encode(series, mapping):
    for v in series.unique():
        if v not in mapping:
            mapping[v] = len(mapping)
    return series.map(mapping).to_numpy(dtype="int32")

In [14]:
users, songs, plays = [], [], []
t0 = time.time()

In [15]:
reader = pd.read_csv(
    PATH, sep="\t", header=None,
    names=["user", "song", "plays"],
    dtype={"plays": "Int32"},
    chunksize=2_000_000,
)

In [16]:
for i, chunk in enumerate(reader):
    users.append(encode(chunk["user"], user_map))
    songs.append(encode(chunk["song"], song_map))
    plays.append(chunk["plays"].to_numpy())
    print(i, len(user_map), len(song_map), f"{time.time()-t0:.0f}s")

0 41756 193683 3s
1 83296 243366 7s
2 125473 272507 10s
3 167503 292756 13s
4 209542 307782 17s
5 251250 319181 20s
6 258348 320614 21s


In [17]:
users = np.concatenate(users)
songs = np.concatenate(songs)
plays = np.concatenate(plays)
print(users.shape, songs.shape, plays.shape)

(12335820,) (12335820,) (12335820,)


In [18]:
np.savez("data/triplets_int.npz", user=users, song=songs, plays=plays)
with open("data/id_maps.pkl", "wb") as f:
    pickle.dump((list(user_map), list(song_map)), f)

In [19]:
d = np.load("data/triplets_int.npz")
users, songs, plays = d["user"], d["song"], d["plays"]
print(users.shape, songs.shape, plays.shape)

(12335820,) (12335820,) (12335820,)


In [20]:
print("min plays:", plays.min())
print("max plays:", plays.max())
print("mean plays:", plays.mean())
print("median plays:", np.median(plays))
print("share of rows with plays == 1:", (plays == 1).mean())

min plays: nan
max plays: nan
mean plays: nan
median plays: nan
share of rows with plays == 1: 0.5932314187463825


In [21]:
qs = [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
print(dict(zip(qs, np.quantile(plays, qs))))

for t in [2, 5, 10, 50, 100, 1000]:
    print(f"plays >= {t}: {(plays >= t).sum():>10,}  ({(plays >= t).mean():.4%})")

{0.5: np.float64(nan), 0.75: np.float64(nan), 0.9: np.float64(nan), 0.95: np.float64(nan), 0.99: np.float64(nan), 0.999: np.float64(nan)}
plays >= 2:  5,017,823  (40.6769%)
plays >= 5:  1,855,777  (15.0438%)
plays >= 10:    634,700  (5.1452%)
plays >= 50:     27,407  (0.2222%)
plays >= 100:      4,977  (0.0403%)
plays >= 1000:         11  (0.0001%)


In [22]:
songs_per_user = np.bincount(users)   # number of rows per user -> distinct songs
n_users, n_songs = len(songs_per_user), songs.max() + 1

qs = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
print("min/max/mean:", songs_per_user.min(), songs_per_user.max(), songs_per_user.mean())
print(dict(zip(qs, np.quantile(songs_per_user, qs))))

for t in [5, 10, 20, 50]:
    print(f"users with < {t} songs: {(songs_per_user < t).mean():.2%}")
    
print("density:", len(users) / (int(n_users) * int(n_songs)))

min/max/mean: 1 1651 47.74885038784895
{0.01: np.float64(10.0), 0.1: np.float64(12.0), 0.25: np.float64(16.0), 0.5: np.float64(27.0), 0.75: np.float64(56.0), 0.9: np.float64(106.0), 0.99: np.float64(289.0)}
users with < 5 songs: 0.00%
users with < 10 songs: 0.00%
users with < 20 songs: 35.13%
users with < 50 songs: 71.51%
density: 0.0001489293991773564


In [23]:
users_per_song = np.bincount(songs)   # number of distinct listeners per song

print("min/max/mean:", users_per_song.min(), users_per_song.max(), users_per_song.mean())
print(dict(zip(qs, np.quantile(users_per_song, qs))))

for t in [2, 5, 10, 50]:
    print(f"songs with < {t} listeners: {(users_per_song < t).mean():.2%}")

top = np.sort(users_per_song)[::-1]
print("share of all rows held by top 100 songs:", top[:100].sum() / len(songs))
print("share of all rows held by top 1% of songs:", top[:len(top)//100].sum() / len(songs))

min/max/mean: 1 27812 38.47561241867167
{0.01: np.float64(1.0), 0.1: np.float64(1.0), 0.25: np.float64(2.0), 0.5: np.float64(5.0), 0.75: np.float64(18.0), 0.9: np.float64(71.0), 0.99: np.float64(570.0)}
songs with < 2 listeners: 19.27%
songs with < 5 listeners: 46.94%
songs with < 10 listeners: 63.70%
songs with < 50 listeners: 86.87%
share of all rows held by top 100 songs: 0.07056815031347734
share of all rows held by top 1% of songs: 0.3494604331126751


In [24]:
for t in [2, 5, 10, 20, 50]:
    keep = users_per_song >= t
    print(f"min {t:>2} listeners: songs kept {keep.sum():>7,} ({keep.mean():.1%}), "
          f"rows kept {keep[songs].mean():.1%}")

min  2 listeners: songs kept 258,832 (80.7%), rows kept 99.5%
min  5 listeners: songs kept 170,104 (53.1%), rows kept 97.5%
min 10 listeners: songs kept 116,373 (36.3%), rows kept 94.6%
min 20 listeners: songs kept  77,101 (24.0%), rows kept 90.3%
min 50 listeners: songs kept  42,083 (13.1%), rows kept 81.4%


In [25]:
for t in [10, 20, 50]:
    row_mask = (users_per_song >= t)[songs]
    spu = np.bincount(users[row_mask], minlength=int(n_users))
    print(f"min {t:>2} listeners: users with 0 songs {(spu == 0).sum():>6,}, "
          f"< 10 songs {(spu < 10).mean():.1%}, median songs {np.median(spu):.0f}")

min 10 listeners: users with 0 songs     18, < 10 songs 2.6%, median songs 26
min 20 listeners: users with 0 songs     64, < 10 songs 4.8%, median songs 25
min 50 listeners: users with 0 songs    213, < 10 songs 9.6%, median songs 23


In [26]:
total_plays = np.bincount(users, weights=plays)   # total plays per user

top = np.argsort(plays)[-10:][::-1]               # 10 rows with the highest play counts
for i in top:
    u, s = users[i], songs[i]
    print(f"plays {plays[i]:>5}  song listeners {users_per_song[s]:>6}  "
          f"user songs {songs_per_user[u]:>4}  user total plays {total_plays[u]:>8.0f}")

plays   nan  song listeners      1  user songs    1  user total plays      nan
plays 2213.0  song listeners    453  user songs  325  user total plays     3657
plays 2130.0  song listeners     55  user songs   16  user total plays     2212
plays 1890.0  song listeners      5  user songs   19  user total plays     2053
plays 1739.0  song listeners    126  user songs   25  user total plays     2205
plays 1729.0  song listeners   6190  user songs   32  user total plays     2329
plays 1539.0  song listeners     10  user songs   27  user total plays     1738
plays 1410.0  song listeners    740  user songs   16  user total plays     1671
plays 1409.0  song listeners    162  user songs   26  user total plays     2598
plays 1266.0  song listeners  11727  user songs   12  user total plays     1354


In [27]:
n_users, n_songs = int(users.max()) + 1, int(songs.max()) + 1

def filter_iter(min_song, min_user):
    mask = np.ones(len(users), dtype=bool)
    it = 0
    while True:
        song_cnt = np.bincount(songs[mask], minlength=n_songs)
        user_cnt = np.bincount(users[mask], minlength=n_users)
        new = mask & (song_cnt >= min_song)[songs] & (user_cnt >= min_user)[users]
        it += 1
        if new.sum() == mask.sum():
            return new, it
        mask = new

for min_song, min_user in [(20, 10), (50, 10), (20, 20)]:
    m, it = filter_iter(min_song, min_user)
    print(f"songs>={min_song}, users>={min_user}: iters {it}, "
          f"rows {m.sum():,} ({m.mean():.1%}), "
          f"users {len(np.unique(users[m])):,}, songs {len(np.unique(songs[m])):,}")

songs>=20, users>=10: iters 7, rows 11,022,022 (89.3%), users 245,805, songs 76,097
songs>=50, users>=10: iters 9, rows 9,809,582 (79.5%), users 232,752, songs 41,077
songs>=20, users>=20: iters 9, rows 9,666,854 (78.4%), users 155,006, songs 70,901


In [28]:
mask, it = filter_iter(100, 10)

old_users, new_users = np.unique(users[mask], return_inverse=True)
old_songs, new_songs = np.unique(songs[mask], return_inverse=True)

u = new_users.astype("int32")
s = new_songs.astype("int32")
p = plays[mask]

print("rows:", len(u))
print("users:", len(old_users), " max new user id + 1:", u.max() + 1)
print("songs:", len(old_songs), " max new song id + 1:", s.max() + 1)

rows: 8378143
users: 216305  max new user id + 1: 216305
songs: 22805  max new song id + 1: 22805


In [29]:
np.savez("data/filtered_100_10.npz",
         user=u, song=s, plays=p,
         old_user=old_users, old_song=old_songs)

## Testing with the filtered data

In [30]:
d = np.load("data/filtered_100_10.npz")
u, s, p = d["user"], d["song"], d["plays"]

print("rows:", len(u))
print("users:", u.max() + 1, " songs:", s.max() + 1)
print("min songs per user:", np.bincount(u).min())
print("min listeners per song:", np.bincount(s).min())

rows: 8378143
users: 216305  songs: 22805
min songs per user: 10
min listeners per song: 100
